# 07_test_sectionizer

Test text sectionization and masking for Phase 3 agentic pipeline.

This notebook validates:
1. Text sectionization into assessment_plan, problems, meds, labs, other
2. Non-cue text extraction (concatenated sections)
3. Trigger sentence masking (removing sentences containing trigger)

In [ ]:
# Setup
import sys
from pathlib import Path

# Resolve project root
CWD = Path.cwd()
if (CWD / 'configs').exists() and (CWD / 'src').exists():
    PROJECT_ROOT = CWD
elif CWD.name == 'notebooks' and (CWD.parent / 'configs').exists():
    PROJECT_ROOT = CWD.parent
else:
    cur = CWD
    PROJECT_ROOT = CWD
    while cur != cur.parent:
        if (cur / 'configs').exists() and (cur / 'src').exists():
            PROJECT_ROOT = cur
            break
        cur = cur.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.sectionizer import sectionize_note

print('Project root:', PROJECT_ROOT)

In [ ]:
# Test samples
test_samples = [
    {
        'text': 'Patient reports daily cocaine use. Assessment: Active substance abuse. Plan: Refer to addiction services.',
        'trigger': 'cocaine'
    },
    {
        'text': 'Problems: Drug overdose. Assessment and Plan: Patient has history of heroin use. Medications: None.',
        'trigger': 'heroin'
    },
    {
        'text': 'Chief Complaint: Patient denies current drug use. History: Past cocaine dependence. Labs: Normal.',
        'trigger': 'cocaine'
    }
]

print(f'Testing {len(test_samples)} samples')

In [ ]:
# Test sectionization
for i, sample in enumerate(test_samples, 1):
    print(f"{'='*80}")
    print(f"Sample {i}")
    print(f"{'='*80}")
    print(f"Original text: {sample['text']}")
    print(f"Trigger: {sample['trigger']}")
    
    result = sectionize_note(
        sample['text'],
        sample['trigger'],
        use_sections=['assessment_plan', 'problems', 'meds', 'labs'],
        mask_trigger=True
    )
    
    print(f"Sections:")
    for section_name, section_text in result['sections'].items():
        if section_text:
            print(f"  - {section_name}: {section_text[:80]}...")
    
    print(f"Non-cue text: {result['non_cue_text'][:100]}...")
    print(f"Masked note: {result['masked_note']}")
    
    # Verify trigger is not in masked note
    if sample['trigger'].lower() in result['masked_note'].lower():
        print(f"  ⚠️  Warning: Trigger '{sample['trigger']}' still found in masked note!")
    else:
        print(f"  ✅ Trigger '{sample['trigger']}' successfully masked")